In [1]:
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import (
    ChatPromptTemplate,PromptTemplate,
    SystemMessagePromptTemplate,MessagesPlaceholder,
    HumanMessagePromptTemplate, FewShotChatMessagePromptTemplate
)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.memory import ConversationBufferMemory,ChatMessageHistory
from langchain_classic.chains import ConversationalRetrievalChain






import gradio as gr
import warnings

from config import get_llm   # must return a LangChain-compatible LLM

warnings.filterwarnings("ignore") # Backup embeddings


In [2]:
loader = PyPDFLoader("aiayn.pdf")
docs = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=150,
    chunk_overlap=20
)
chunks = splitter.split_documents(docs)

vectordb = Chroma.from_documents(
        chunks,
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    )

retriever = vectordb.as_retriever(search_kwargs={"k": 5})


In [6]:
llm=ChatOllama(model="llama3", temperature=0.2)

### CONDENSE_PROMPT

In [16]:


CONDENSE_PROMPT_2 = ChatPromptTemplate.from_messages([
    ("system", "Rewrite follow-up questions into standalone questions. Do NOT answer the question."),
    MessagesPlaceholder(variable_name="chat_history"),  # ✅ Use MessagesPlaceholder for list of messages
    ("human", "Follow-up question: {question}\n\nStandalone question:")
])

### Actual Prompt

In [17]:
PROMPT_CHAT_2 = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("""You are a helpful technical assistant. Answer questions using the conversation history and retrieved documents.

IMPORTANT INSTRUCTIONS:
1. First check if the question was already answered in the chat_history 
2. If the answer is in the chat_history, use that answer
3. If not, use Retrieved Documents if they are relevant to the question
4. If question is about any topic which is not in Retrieved Documents or in chat_history then just say 'I cannot answer this question'
5. Give a clear, direct answer without mentioning irrelevant information"""),
    
    MessagesPlaceholder(variable_name="chat_history"),
    
    HumanMessagePromptTemplate.from_template("""Retrieved Documents:
{context}

Current Question: {question}""")
])

In [27]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def build_chain_with_condense():
    # Step 1: Condense question
    standalone_chain = (
        {
            "question": lambda x: x["question"],
            "chat_history": lambda x: x.get("chat_history", [])
        }
        | CONDENSE_PROMPT_2
        | llm
        | StrOutputParser()
    )
    
    def get_inputs(x):
        # Get standalone question once
        standalone_q = standalone_chain.invoke(x)
        
        # Retrieve documents using standalone question
        docs = retriever.invoke(standalone_q)
        context = format_docs(docs)
        
        return {
            "context": context,
            "question": standalone_q,  # Use the condensed question
            "chat_history": x.get("chat_history", [])
        } 
        
    chain = (
        get_inputs
        | PROMPT_CHAT_2
        | llm
        | StrOutputParser()
    )
    
    return chain

In [38]:
chain = build_chain_with_condense()
history = []

In [32]:
def ask(chain, question, history):
    response = chain.invoke({
        "question": question,
        "chat_history": history
    })

    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=response))

    return response


In [40]:
ask(chain, 'who created it', history)

'Based on the Retrieved Documents, I can answer that Ashish, with Illia, designed and implemented the first Transformer models.'

In [41]:
history

[HumanMessage(content='what is transformer', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Based on the Retrieved Documents, I can tell you that a Transformer is a type of transduction model that allows for significantly more parallelization and has reached a new state of the art. It's the first model to rely on self-attention mechanisms, which makes it more difficult to learn dependencies between distant positions.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='who created it', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Based on the Retrieved Documents, I can answer that Ashish, with Illia, designed and implemented the first Transformer models.', additional_kwargs={}, response_metadata={})]

In [48]:
formatted = PROMPT_CHAT_2.format_messages(
    chat_history=history,
    question="why is it better than rnn",
    context = 'q'
)

for m in formatted:
    print(type(m).__name__, ":", m.content)

SystemMessage : You are a helpful technical assistant. Answer questions using the conversation history and retrieved documents.

IMPORTANT INSTRUCTIONS:
1. First check if the question was already answered in the chat_history 
2. If the answer is in the chat_history, use that answer and expand on it
3. If not, use Retrieved Documents if they are relevant to the question
4. If question is about any topic which is not in Retrieved Documents or in chat_history then just say 'I cannot answer this question'
5. Give a clear, direct answer without mentioning irrelevant information
HumanMessage : what is transformer
AIMessage : Based on the Retrieved Documents, I can tell you that a Transformer is a type of transduction model that allows for significantly more parallelization and has reached a new state of the art. It's the first model to rely on self-attention mechanisms, which makes it more difficult to learn dependencies between distant positions.
HumanMessage : who created it
AIMessage : Ba